# CSCI 4253 / 5253 - Lab #4 - Patent Problem with Spark DataFrames
<div>
 <h2> CSCI 4283 / 5253 
  <IMG SRC="https://www.colorado.edu/cs/profiles/express/themes/cuspirit/logo.png" WIDTH=50 ALIGN="right"/> </h2>
</div>

This [Spark cheatsheet](https://s3.amazonaws.com/assets.datacamp.com/blog_assets/PySpark_SQL_Cheat_Sheet_Python.pdf) is useful as is [this reference on doing joins in Spark dataframe](http://www.learnbymarketing.com/1100/pyspark-joins-by-example/).

The [DataBricks company has one of the better reference manuals for PySpark](https://docs.databricks.com/spark/latest/dataframes-datasets/index.html) -- they show you how to perform numerous common data operations such as joins, aggregation operations following `groupBy` and the like.

In [1]:
from pyspark.sql import SparkSession
from pyspark import StorageLevel

The following aggregation functions may be useful -- [these can be used to aggregate results of `groupby` operations](https://docs.databricks.com/spark/latest/dataframes-datasets/introduction-to-dataframes-python.html#example-aggregations-using-agg-and-countdistinct). More documentation is at the [PySpark SQL Functions manual](https://spark.apache.org/docs/2.3.0/api/python/pyspark.sql.html#module-pyspark.sql.functions). Feel free to use other functions from that library.

In [2]:
from pyspark.sql.functions import col, count, coalesce, lit, sum as spark_sum

Create our session as described in the tutorials

In [3]:
spark = (SparkSession.builder
         .appName("Lab4-Dataframe")
         .master("local[4]")
         .config("spark.sql.shuffle.partitions", "16")
         .config("spark.ui.showConsoleProgress", "false")
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/17 21:15:48 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Read in the citations and patents data and check that the data makes sense. Note that unlike in the RDD solution, the data is automatically inferred to be Integer() types.

In [4]:
citations = spark.read.load('cite75_99.txt.gz',
            format="csv", sep=",", header=True,
            compression="gzip",
            inferSchema="true")

In [5]:
citations.show(5)

+-------+-------+
| CITING|  CITED|
+-------+-------+
|3858241| 956203|
|3858241|1324234|
|3858241|3398406|
|3858241|3557384|
|3858241|3634889|
+-------+-------+
only showing top 5 rows



In [6]:
patents = spark.read.load('apat63_99.txt.gz',
            format="csv", sep=",", header=True,
            compression="gzip",
            inferSchema="true")

In [7]:
patents.show(5)

+-------+-----+-----+-------+-------+-------+--------+-------+------+------+---+------+-----+--------+--------+-------+--------+--------+--------+--------+--------+--------+--------+
| PATENT|GYEAR|GDATE|APPYEAR|COUNTRY|POSTATE|ASSIGNEE|ASSCODE|CLAIMS|NCLASS|CAT|SUBCAT|CMADE|CRECEIVE|RATIOCIT|GENERAL|ORIGINAL|FWDAPLAG|BCKGTLAG|SELFCTUB|SELFCTLB|SECDUPBD|SECDLWBD|
+-------+-----+-----+-------+-------+-------+--------+-------+------+------+---+------+-----+--------+--------+-------+--------+--------+--------+--------+--------+--------+--------+
|3070801| 1963| 1096|   NULL|     BE|   NULL|    NULL|      1|  NULL|   269|  6|    69| NULL|       1|    NULL|    0.0|    NULL|    NULL|    NULL|    NULL|    NULL|    NULL|    NULL|
|3070802| 1963| 1096|   NULL|     US|     TX|    NULL|      1|  NULL|     2|  6|    63| NULL|       0|    NULL|   NULL|    NULL|    NULL|    NULL|    NULL|    NULL|    NULL|    NULL|
|3070803| 1963| 1096|   NULL|     US|     IL|    NULL|      1|  NULL|     2|  6|    6

## Attach states and count same-state citations

Select US patents with a nonempty state. Join citations to this lookup twice: once for the cited patent and once for the citing patent. Missing patents and missing states cannot contribute to the count. Keep matching states and group by the citing patent.

Finally, left join the counts onto the original patent table and fill missing counts with zero. This preserves patents with no citations, unknown states, and no same-state matches.

In [8]:
def augment_patents(patents, citations):
    states = patents.filter(
        (col("COUNTRY") == "US") & col("POSTATE").isNotNull() & (col("POSTATE") != "")
    ).select("PATENT", "POSTATE")
    cited_states = states.select(
        col("PATENT").alias("CITED"), col("POSTATE").alias("CITED_STATE")
    )
    citing_states = states.select(
        col("PATENT").alias("CITING"), col("POSTATE").alias("CITING_STATE")
    )
    matches = (citations.join(cited_states, "CITED")
               .join(citing_states, "CITING")
               .filter(col("CITED_STATE") == col("CITING_STATE")))
    counts = matches.groupBy("CITING").agg(count("*").alias("SAME_STATE"))
    return (patents.join(counts, patents.PATENT == counts.CITING, "left")
            .drop("CITING")
            .withColumn("SAME_STATE", coalesce(col("SAME_STATE"), lit(0))))

## Check boundary cases

This small example includes a same-state citation, a different-state citation, empty and null states, a non-US patent with a state label, missing patent IDs, and patents without citations. Only patent 1 should receive a count of one.

In [9]:
example_patents = spark.createDataFrame([
    (1, "US", "NY"), (2, "US", "NY"), (3, "US", "CA"),
    (4, "US", None), (5, "US", ""), (6, "GB", "NY"), (7, "US", "TX")
], "PATENT int, COUNTRY string, POSTATE string")
example_citations = spark.createDataFrame([
    (1, 2), (1, 3), (1, 4), (1, 5), (1, 6), (1, 999),
    (4, 5), (5, 4), (6, 2), (999, 2)
], "CITING int, CITED int")
actual = {r.PATENT: r.SAME_STATE for r in augment_patents(
    example_patents, example_citations).collect()}
assert actual == {1: 1, 2: 0, 3: 0, 4: 0, 5: 0, 6: 0, 7: 0}
print("Boundary-case checks passed.")

Boundary-case checks passed.


## Full dataset result

Run on every citation and patent, without sampling. Cache the augmented table because the ranking and validation reuse it. Sort by descending same-state count, then ascending patent number to make ties reproducible.

In [10]:
augmented = augment_patents(patents, citations).persist(StorageLevel.MEMORY_AND_DISK)
patent_count = patents.count()
citation_count = citations.count()
assert augmented.count() == patent_count
print(f"Patents: {patent_count:,}; citations: {citation_count:,}")
augmented.orderBy(col("SAME_STATE").desc(), col("PATENT")).show(10, truncate=False)

Patents: 2,923,922; citations: 16,522,438


+-------+-----+-----+-------+-------+-------+--------+-------+------+------+---+------+-----+--------+--------+-------+--------+--------+--------+--------+--------+--------+--------+----------+
|PATENT |GYEAR|GDATE|APPYEAR|COUNTRY|POSTATE|ASSIGNEE|ASSCODE|CLAIMS|NCLASS|CAT|SUBCAT|CMADE|CRECEIVE|RATIOCIT|GENERAL|ORIGINAL|FWDAPLAG|BCKGTLAG|SELFCTUB|SELFCTLB|SECDUPBD|SECDLWBD|SAME_STATE|
+-------+-----+-----+-------+-------+-------+--------+-------+------+------+---+------+-----+--------+--------+-------+--------+--------+--------+--------+--------+--------+--------+----------+
|5959466|1999 |14515|1997   |US     |CA     |5310    |2      |NULL  |326   |4  |46    |159  |0       |1.0     |NULL   |0.6186  |NULL    |4.8868  |0.0455  |0.044   |NULL    |NULL    |125       |
|5983822|1999 |14564|1998   |US     |TX     |569900  |2      |NULL  |114   |5  |55    |200  |0       |0.995   |NULL   |0.7201  |NULL    |12.45   |0.0     |0.0     |NULL    |NULL    |103       |
|6008204|1999 |14606|1998   |U

## Verify the reference example and totals

The supplied data gives patent 6009554 eight same-state citations: eight cited patents are from NY and one is from Great Britain. The README describes six, so its example does not match these files. Patent 5959466 leads with 125, matching the reference figure. The total counts and squared counts provide additional comparisons with the RDD solution.

Several patents tie at 90 near the cutoff. Sorting tied patents by ID can select a different set of ten than the reference figure without changing the counts.

In [11]:
assert augmented.filter(col("PATENT") == 6009554).first().SAME_STATE == 8
leader = augmented.orderBy(col("SAME_STATE").desc(), col("PATENT")).first()
assert (leader.PATENT, leader.SAME_STATE) == (5959466, 125)
summary = augmented.agg(
    spark_sum("SAME_STATE").alias("total_same_state"),
    spark_sum(col("SAME_STATE") * col("SAME_STATE")).alias("sum_squared_counts")
).first()
print(summary.asDict())
print("Full-dataset checks passed.")

{'total_same_state': 1488330, 'sum_squared_counts': 9103680}
Full-dataset checks passed.


In [13]:
augmented.unpersist()
spark.stop()